In [2]:
import os
os.chdir('/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning')
print("Current working directory: {0}".format(os.getcwd()))

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload
%autoreload 2

Current working directory: /Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

from typing import List
from tqdm import tqdm
import math


from batchbald_redux import (
        active_learning,
        batchbald,
        consistent_mc_dropout,
        joint_entropy,
        repeated_mnist,
    )

In [4]:
class BayesianCNN(consistent_mc_dropout.BayesianModule):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=4)
        self.conv1_drop = consistent_mc_dropout.ConsistentMCDropout2d(0.25)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=4)
        self.conv2_drop = consistent_mc_dropout.ConsistentMCDropout2d(0.25)
        self.fc1 = nn.Linear(1024, 128)
        self.fc1_drop = consistent_mc_dropout.ConsistentMCDropout(0.25)
        self.fc2 = nn.Linear(128, num_classes)

    def mc_forward_impl(self, input: torch.Tensor):
        input = F.relu(F.max_pool2d(self.conv1_drop(self.conv1(input)), 2))
        input = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(input)), 2))
        input = input.view(-1, 1024)
        input = F.relu(self.fc1_drop(self.fc1(input)))
        input = self.fc2(input)
        input = F.log_softmax(input, dim=1)

        return input

In [5]:
class BayesianCNN(consistent_mc_dropout.BayesianModule):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=5)
        self.conv1_drop = consistent_mc_dropout.ConsistentMCDropout2d()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        self.conv2_drop = consistent_mc_dropout.ConsistentMCDropout2d()
        self.fc1 = nn.Linear(1024, 128)
        self.fc1_drop = consistent_mc_dropout.ConsistentMCDropout()
        self.fc2 = nn.Linear(128, num_classes)

    def mc_forward_impl(self, input: torch.Tensor):
        input = F.relu(F.max_pool2d(self.conv1_drop(self.conv1(input)), 2))
        input = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(input)), 2))
        input = input.view(-1, 1024)
        input = F.relu(self.fc1_drop(self.fc1(input)))
        input = self.fc2(input)
        input = F.log_softmax(input, dim=1)

        return input

In [6]:
num_classes = 10
num_inference_samples = 10
num_test_inference_samples = 5
num_samples = 100000  # Total number of samples

test_batch_size = 512  # Test Loader Batch size
batch_size = 64 # Train loader Batch size
scoring_batch_size = 128  # Pool Loader Batch size
training_iterations = 4096 * 6

use_cuda = torch.cuda.is_available()
print(f"use_cuda: {use_cuda}")
device = "cuda" if use_cuda else "cpu"
kwargs = {"num_workers": 0, "pin_memory": True} if use_cuda else {}

use_cuda: False


In [7]:
# get an active learning dataset
mnist_train = MNIST(root='./data/raw', train=True, transform=transforms.ToTensor())
mnist_test = MNIST(root='./data/raw', train=False, transform=transforms.ToTensor())
test_loader = DataLoader(mnist_test, batch_size=512, shuffle=False, num_workers=0)

active_learning_data = active_learning.ActiveLearningData(mnist_train)

In [8]:
def entire_batchbald_run(traindata: Dataset,
                         testdata: Dataset,
                         initial_samples: list,
                         query_size = int,
                         num_queries = int,
                         type = 'batchbald'             
                         ):
    
    """ Performs an entire active learning training using BatchBALD acquisition strategy. """
    active_learning_data = active_learning.ActiveLearningData(traindata)
    active_learning_data.acquire(
            initial_samples
        )  
    
    print(f'Length of trainset: {len(active_learning_data.training_dataset)}')
    print(f'Length of poolset: {len(active_learning_data.pool_dataset)}')
    
    train_loader = DataLoader(
            active_learning_data.training_dataset,
            sampler=active_learning.RandomFixedLengthSampler(
                active_learning_data.training_dataset, training_iterations
            ),
            batch_size=batch_size,
            **kwargs,
        )

    pool_loader = DataLoader(
                active_learning_data.pool_dataset,
                batch_size=scoring_batch_size,
                shuffle=False,
                **kwargs,
            )
    
    test_loader = DataLoader(
                testdata,
                batch_size=test_batch_size,
                shuffle=False,
                **kwargs,
            )
        
    test_accs = []
    test_loss = []
    added_indices = []
    
    pbar = tqdm(
            initial=len(active_learning_data.training_dataset),
            #total=max_training_samples,
            desc="Training Set Size",
        )
    
    for query in range(num_queries):
        model = BayesianCNN(num_classes).to(device=device)  # initialise model
        optimizer = torch.optim.Adam(model.parameters())

        model.train()

        # Train
        for data, target in tqdm(train_loader, desc="Training", leave=False):
            data = data.to(device=device)
            target = target.to(device=device)
            
            optimizer.zero_grad()

            prediction = model(data, 1).squeeze(1)

            loss = F.nll_loss(prediction, target)

            loss.backward()
            optimizer.step()

        # Test
        loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in tqdm(test_loader, desc="Testing", leave=False):
                data = data.to(device=device)
                target = target.to(device=device)

                prediction = torch.logsumexp(model(data, num_test_inference_samples), dim=1) - math.log(
                    num_test_inference_samples
                )
                loss += F.nll_loss(prediction, target, reduction="sum")

                prediction = prediction.max(1)[1]
                correct += prediction.eq(target.view_as(prediction)).sum().item()

        loss /= len(test_loader.dataset)
        test_loss.append(loss)

        percentage_correct = 100.0 * correct / len(test_loader.dataset)
        test_accs.append(percentage_correct)

        print("Test set: Average loss: {:.4f}, Accuracy: ({:.2f}%)".format(loss, percentage_correct))

        # Acquire pool predictions
        N = len(active_learning_data.pool_dataset)
        logits_N_K_C = torch.empty(
            (N, num_inference_samples, num_classes),
            dtype=torch.double,
            pin_memory=use_cuda,
        )

        with torch.no_grad():
            model.eval()

            for i, (data, _) in enumerate(tqdm(pool_loader, desc="Evaluating Acquisition Set", leave=False)):
                data = data.to(device=device)

                lower = i * pool_loader.batch_size
                upper = min(lower + pool_loader.batch_size, N)
                logits_N_K_C[lower:upper].copy_(model(data, num_inference_samples).double(), non_blocking=True)

        print(logits_N_K_C.shape)
        with torch.no_grad():
            if type == "batchbald":
                candidate_batch = batchbald.get_batchbald_batch(
                    logits_N_K_C,
                    query_size,
                    num_samples,
                    dtype=torch.double,
                    device=device,  # Returns the indices and scores(Mutual Information) for the batch selected by Batchbald/BALD Strategy.
                )

            elif type == "bald":
                candidate_batch = batchbald.get_bald_batch(
                    logits_N_K_C,
                    query_size,
                    dtype=torch.double,
                    device=device,
                )

        dataset_indices = active_learning_data.get_dataset_indices(
            candidate_batch.indices
        )  # Returns indices for candidate batch

        print("Dataset indices: ", dataset_indices)

        active_learning_data.acquire(candidate_batch.indices)  # add the new indices to training dataset
        added_indices.append(dataset_indices)
        pbar.update(len(dataset_indices))
        print(candidate_batch.scores)
        

In [9]:
num_initial_samples = 20

initial_samples = active_learning.get_balanced_sample_indices(
            repeated_mnist.get_targets(mnist_train),
            num_classes=num_classes,
            n_per_digit=num_initial_samples / num_classes,
        )

entire_batchbald_run(mnist_train, mnist_test, initial_samples, query_size=4, num_queries=10)

Length of trainset: 20
Length of poolset: 59980


Training Set Size: 20it [00:00, ?it/s]

Test set: Average loss: 1.9637, Accuracy: (61.18%)


Conditional Entropy:   0%|          | 0/59980 [00:00<?, ?it/s]

BatchBALD:   0%|          | 0/4 [00:00<?, ?it/s]

...

ExactJointEntropy.compute_batch:   0%|          | 0/59980 [00:00<?, ?it/s]

Training Set Size: 24it [01:19, 19.97s/it]

Dataset indices:  [34481 54161  9329 37829]
[1.233221051434847, 1.8897789324951917, 2.1386309251739544, 2.2401842564744707]


Test set: Average loss: 1.6588, Accuracy: (66.20%)


KeyboardInterrupt: 

In [9]:
candidate_batch

NameError: name 'candidate_batch' is not defined